# V3 Data Loading + Feature Engineering

Pipeline for V3 dataset (35-dim frames with integral_pos_err):
1. Drop formation one-hot (4 cols)
2. Yaw cos/sin transform
3. Z-score normalize (train split only)
4. Max-abs scale labels
5. Save normalization stats

## 1. Load Splits

In [ ]:
import sys, torch
sys.path.insert(0, '../src')
from pathlib import Path
from dataloader import SplitDataset, engineer_x, DatasetNormalizer

TRAIN = r"../datasets/setpoint_V3_mixed_formations_train.pt"
VAL   = r"../datasets/setpoint_V3_mixed_formations_val.pt"
TEST  = r"../datasets/setpoint_V3_mixed_formations_test.pt"

train_ds = SplitDataset(TRAIN)
val_ds   = SplitDataset(VAL)
test_ds  = SplitDataset(TEST)

print(f"train: {len(train_ds)}  val: {len(val_ds)}  test: {len(test_ds)}")
s = train_ds[0]
print(f"x: {tuple(s.x.shape)}  target: {tuple(s.target.shape)}  edges: {tuple(s.edge_attr.shape)}")

## 2. Feature Layout

In [ ]:
RAW_FRAME  = 35
YAW_IDX    = 25
PROC_FRAME = 32   # 35 - 4(formation) - 1(yaw) + 2(cos/sin)

print(f"Raw stacked:  {RAW_FRAME * 2}")
print(f"Proc stacked: {PROC_FRAME * 2}")
print(f"cos/sin indices: [25, 26, {25+PROC_FRAME}, {26+PROC_FRAME}]")

## 3. Verify Engineering

In [ ]:
sample = train_ds[0]
x_eng = engineer_x(sample.x)
print(f"Raw: {tuple(sample.x.shape)} → Engineered: {tuple(x_eng.shape)}")
assert x_eng.shape[1] == 64, f"Expected 64, got {x_eng.shape[1]}"

## 4. Fit Normalizer

In [ ]:
CONFIG = {
    "raw_frame_dim": 35,
    "yaw_idx_in_frame": 25,
    "cos_sin_indices": [25, 26, 57, 58],
    "yaw_quantile": 0.99,
}

normalizer = DatasetNormalizer.fit(train_ds, CONFIG)
print(f"y_scale: {normalizer.y_scale}")

## 5. Save Stats

In [ ]:
out = Path('../results'); out.mkdir(exist_ok=True)
normalizer.save(out / 'normalization_stats.pt')

# Also copy to checkpoints for inference
ckpt = Path('../checkpoints'); ckpt.mkdir(exist_ok=True)
normalizer.save(ckpt / 'normalization_stats.pt')
print('Saved.')